In [26]:
# Importar CSV
import csv
import pandas as pd

file = open('datahvsmio.csv', 'r')
reader = csv.reader(file)
data = list(reader)

df = pd.DataFrame(data[1:], columns=data[0])
print(df.head())

                      Vertical_Dissonance_Ratio Standard_Triads  \
0   00001REARM_OG.mid                    0.1429            0.25   
1  00003REARM_MIO.mid                    0.2912             0.0   
2  00004REARM_MIO.mid                    0.1398             0.5   
3  00005REARM_MIO.mid                    0.1589            0.25   
4  00002REARM_MIO.mid                    0.3501          0.1429   

  Diminished_and_Augmented_Triads Dominant_Seventh_Chords Seventh_Chords  \
0                             0.0                     0.0           0.75   
1                             0.0                     0.0           0.75   
2                             0.0                     0.0            0.0   
3                             0.0                     0.0            0.5   
4                             0.0                  0.2857         0.4286   

  Non-Standard_Chords Complex_Chords  
0                 0.0            0.0  
1                0.25           0.25  
2                 0.5  

In [27]:
# Dividir datos por los nombres, formato 00001REARM_MIO, 00001REARM_OG y 00001SIMPLE_OG con numeros del 00001 al 000006
df_rearm_og = df[df[''].str.contains('REARM_OG')]
df_rearm_mio = df[df[''].str.contains('REARM_MIO')]
df_simple_og = df[df[''].str.contains('SIMPLE_OG')]

In [28]:
# Pasar las columnas a tipo float y calcular media y desviacion estandar, menos la columna de nombres
mean_rearm_og = df_rearm_og.iloc[:, 1:].astype(float).mean()
std_rearm_og = df_rearm_og.iloc[:, 1:].astype(float).std()
mean_rearm_mio = df_rearm_mio.iloc[:, 1:].astype(float).mean()
std_rearm_mio = df_rearm_mio.iloc[:, 1:].astype(float).std()
mean_simple_og = df_simple_og.iloc[:, 1:].astype(float).mean()
std_simple_og = df_simple_og.iloc[:, 1:].astype(float).std()


# Imprimir por pantalla como tabla con +- desviacion estandar
print("REARM OG:")
for col in mean_rearm_og.index:
    print(f"{col}: {mean_rearm_og[col]:.4f} ± {std_rearm_og[col]:.4f}")
print("\nREARM MIO:")
for col in mean_rearm_mio.index:
    print(f"{col}: {mean_rearm_mio[col]:.4f} ± {std_rearm_mio[col]:.4f}")
print("\nSIMPLE OG:")
for col in mean_simple_og.index:
    print(f"{col}: {mean_simple_og[col]:.4f} ± {std_simple_og[col]:.4f}")


REARM OG:
Vertical_Dissonance_Ratio: 0.3140 ± 0.2983
Standard_Triads: 0.4375 ± 0.4165
Diminished_and_Augmented_Triads: 0.0000 ± 0.0000
Dominant_Seventh_Chords: 0.0417 ± 0.0645
Seventh_Chords: 0.1667 ± 0.2923
Non-Standard_Chords: 0.3958 ± 0.4771
Complex_Chords: 0.2500 ± 0.4183

REARM MIO:
Vertical_Dissonance_Ratio: 0.2360 ± 0.1005
Standard_Triads: 0.2321 ± 0.2278
Diminished_and_Augmented_Triads: 0.0000 ± 0.0000
Dominant_Seventh_Chords: 0.0476 ± 0.1166
Seventh_Chords: 0.3214 ± 0.2962
Non-Standard_Chords: 0.4464 ± 0.1873
Complex_Chords: 0.2143 ± 0.2936

SIMPLE OG:
Vertical_Dissonance_Ratio: 0.0133 ± 0.0327
Standard_Triads: 0.9792 ± 0.0510
Diminished_and_Augmented_Triads: 0.0000 ± 0.0000
Dominant_Seventh_Chords: 0.0208 ± 0.0510
Seventh_Chords: 0.0208 ± 0.0510
Non-Standard_Chords: 0.0000 ± 0.0000
Complex_Chords: 0.0000 ± 0.0000


In [29]:
from mido import MidiFile
from music21 import chord, note
import os

def identify_chords(midi_file):
    mid = MidiFile(midi_file)
    notes_at_time = {}
    
    for track in mid.tracks:
        temp_tick = 0
        for msg in track:
            temp_tick += msg.time
            if msg.type == 'note_on' and msg.velocity > 0:
                # Agrupamos notas por pulsos (480 ticks suele ser una negra)
                time_step = temp_tick // 480 
                if time_step not in notes_at_time:
                    notes_at_time[time_step] = []
                notes_at_time[time_step].append(msg.note)

    identified_chords = []
    
    for t in sorted(notes_at_time.keys()):
        pitches = list(set(notes_at_time[t])) 
        if len(pitches) >= 2:
            c = chord.Chord(pitches)
            
            # Formateo para obtener algo tipo "C Major" o "A Minor"
            name = c.pitchedCommonName.replace("-alpha", "").replace("triad", "").strip()
            
            # Evitar repeticiones consecutivas del mismo acorde
            if not identified_chords or identified_chords[-1] != name:
                identified_chords.append(name)

    return identified_chords

# --- Bloque principal con el filtro de nombre ---

# Listamos archivos que terminan en .mid/.midi Y que NO contienen "simple"
midi_files = [
    f for f in os.listdir() 
    if (f.lower().endswith('.mid') or f.lower().endswith('.midi')) 
    and 'simple' not in f.lower()
]

if not midi_files:
    print("No se encontraron archivos MIDI que cumplan los criterios.")

for midi_file in midi_files:
    print(f"🎵 Procesando: {midi_file}")
    try:
        chords = identify_chords(midi_file)
        if chords:
            print(" -> ".join(chords))
        else:
            print(" No se detectaron acordes (posiblemente solo melodía monofónica).")
    except Exception as e:
        print(f" Error en {midi_file}: {e}")
    print("-" * 40)

🎵 Procesando: 00001REARM_OG.mid
C-major seventh chord -> A-minor seventh chord -> D-minor seventh chord -> G-major
----------------------------------------
🎵 Procesando: 00003REARM_MIO.mid
D-minor-ninth chord -> E-minor seventh chord -> D-minor seventh chord -> E-minor seventh chord
----------------------------------------
🎵 Procesando: 00004REARM_MIO.mid
G-major-second major tetrachord -> F-major -> G-major
----------------------------------------
🎵 Procesando: 00005REARM_MIO.mid
A-minor -> B-half-diminished seventh chord -> E-minor seventh chord -> A-minor seventh chord
----------------------------------------
🎵 Procesando: 00002REARM_MIO.mid
D-minor -> Bb-dominant seventh chord -> A-minor-ninth chord -> Eb-whole-tone tetramirror -> D-minor-ninth chord -> G-dominant seventh chord -> A-major seventh chord
----------------------------------------
🎵 Procesando: 00004REARM_OG.mid
C-major -> E-incomplete dominant-seventh chord -> A-minor -> A-dominant seventh chord -> D-minor -> G-major
-